In [9]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 读取标签表和映射表
labels = pd.read_csv("labels.csv")

# 股票代码强制为字符串，并补齐前导零到6位
mapping = pd.read_csv("Eastmoney_report_pdf_download\\HS300.csv", dtype={"股票代码": str})
mapping["股票代码"] = mapping["股票代码"].str.zfill(6)

# 建立简称 -> 代码 的映射字典
name2code = dict(zip(mapping["股票简称"], mapping["股票代码"]))

# 2. 遍历报告目录，收集文本和对应标签
data = []
base_dir = "reports_txt_by_quarter_cleaned_en"

for quarter in os.listdir(base_dir):
    quarter_path = os.path.join(base_dir, quarter)
    if not os.path.isdir(quarter_path):
        continue
    
    # 标签表里的交易日期格式可能是 2017Q1，而文件夹是 2017_Q1
    quarter_key = quarter.replace("_", "")
    label_row = labels[labels["交易日期"] == quarter_key]
    if label_row.empty:
        continue
    
    for company_name in os.listdir(quarter_path):
        company_path = os.path.join(quarter_path, company_name)
        if not os.path.isdir(company_path):
            continue
        
        # 映射公司简称到股票代码
        if company_name not in name2code:
            print(f"未找到映射: {company_name}")
            continue
        stock_code = name2code[company_name]
        
        # 获取该公司在该季度的标签
        if stock_code in label_row.columns:
            company_label = int(label_row[stock_code].values[0])
        else:
            print(f"标签表中缺少股票代码: {stock_code}")
            continue
        
        # 遍历该公司所有报告文件
        for root, _, files in os.walk(company_path):
            for file in files:
                if file.endswith(".txt"):
                    file_path = os.path.join(root, file)
                    try:
                        with open(file_path, "r", encoding="utf-8") as f:
                            text = f.read().strip()
                        if len(text) > 50:  # 去掉过短的报告
                            data.append({"text": text, "label": company_label})
                    except Exception as e:
                        print(f"跳过文件 {file_path}: {e}")

# 3. 转换为 DataFrame
df = pd.DataFrame(data)

# 4. 划分训练/验证集
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

# 5. 保存为 CSV
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)

print("生成完成: train.csv 和 val.csv")


未找到映射: 比亚迪电子
未找到映射: 中航电测
未找到映射: 中航电测
生成完成: train.csv 和 val.csv


In [12]:
import os
import pandas as pd

# 1. 读取标签表和映射表
labels = pd.read_csv("labels.csv")
mapping = pd.read_csv("Eastmoney_report_pdf_download\\HS300.csv", dtype={"股票代码": str})
mapping["股票代码"] = mapping["股票代码"].str.zfill(6)
name2code = dict(zip(mapping["股票简称"], mapping["股票代码"]))

# 2. 遍历报告目录，收集文本和对应标签
data = []
base_dir = "reports_txt_by_quarter_cleaned_en"

for quarter in sorted(os.listdir(base_dir)):  # 按时间顺序排序
    quarter_path = os.path.join(base_dir, quarter)
    if not os.path.isdir(quarter_path):
        continue
    
    quarter_key = quarter.replace("_", "")
    label_row = labels[labels["交易日期"] == quarter_key]
    if label_row.empty:
        continue
    
    for company_name in os.listdir(quarter_path):
        company_path = os.path.join(quarter_path, company_name)
        if not os.path.isdir(company_path):
            continue
        
        if company_name not in name2code:
            continue
        stock_code = name2code[company_name]
        
        if stock_code not in label_row.columns:
            continue
        company_label = int(label_row[stock_code].values[0])
        
        for root, _, files in os.walk(company_path):
            for file in files:
                if file.endswith(".txt"):
                    file_path = os.path.join(root, file)
                    try:
                        with open(file_path, "r", encoding="utf-8") as f:
                            text = f.read().strip()
                        if len(text) > 50:
                            data.append({"quarter": quarter_key, "text": text, "label": company_label})
                    except Exception as e:
                        print(f"跳过文件 {file_path}: {e}")

# 3. 转换为 DataFrame，按季度排序
df = pd.DataFrame(data)
df = df.sort_values("quarter")

# 4. 时间序列划分：前80%季度作为训练集，后20%季度作为测试集
unique_quarters = df["quarter"].unique()
split_point = int(len(unique_quarters) * 0.8)

train_quarters = unique_quarters[:split_point]
test_quarters = unique_quarters[split_point:]

train_df = df[df["quarter"].isin(train_quarters)]
test_df = df[df["quarter"].isin(test_quarters)]

train_df.to_csv("train_timeseries.csv", index=False)
test_df.to_csv("test_timeseries.csv", index=False)

print(f"生成完成: 训练集 {train_quarters[0]}–{train_quarters[-1]} → 测试集 {test_quarters[0]}–{test_quarters[-1]}")


生成完成: 训练集 2017Q1–2023Q4 → 测试集 2024Q1–2025Q4
